# Coupang Internal Brand Review Analysis (English Version)
Analyzes Coupang product reviews for the company's own diaper product lines.
Extracts satisfaction/dissatisfaction keywords, generates property scores and summaries,
and flags risk-related mentions using GPT-4.

**Sections:** Keyword Extraction → Property Rating + Summary → Risk Detection

**Run:** Update `start` and `end` dates before each batch.

In [ ]:
# pip install googletrans==4.0.0rc1 snowflake-connector-python snowflake-sqlalchemy==1.5.1 openai

In [ ]:
import os
import re
import time
import pandas as pd
import datetime
import openai
import snowflake.connector
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine
from snowflake.connector.pandas_tools import pd_writer

# ── BRAND KEYWORD DICTIONARY ──────────────────────────────────────────────
# Maps each brand to known product line names and aliases (English)
# Note: Original pipeline ran on Korean community data;
# this version uses English-transliterated equivalents for readability.

BRAND_KEYWORDS = {
    'Huggies':  '|'.join(['Huggies', 'MaxDry', 'MagicDry', 'NatureMade',
                           'MagicComfort', 'NatureMadeOrganic', 'NatureSummer']),
    'Pampers':  '|'.join(['Pampers', 'Armoni', 'BabyDry', 'TouchOfNature',
                           'AirChaCha', 'NightPants']),
    'Penelope': '|'.join(['Penelope', 'MiracleAllDay', 'ThinLight']),
    'Mamipoko': '|'.join(['Mamipoko', 'AirFit', 'GoldBreathable', 'LeafGanic']),
    'Bosomi':   '|'.join(['Bosomi', 'WonderByWonder', 'MegaDry', 'RealCottonOrganic']),
    'Kindoh':   '|'.join(['Kindoh', 'UpAndPlay', 'AllDay', 'SlimFit'])
}

TARGET_BRANDS = list(BRAND_KEYWORDS.keys())

# ── LEAKAGE DETECTION KEYWORDS ────────────────────────────────────────────
# Detects posts describing diaper leakage incidents

LEAKAGE_KEYWORDS = '|'.join([
    'leaking', 'leaked', 'leaks',
    'soaked', 'soaking', 'wet',
    'dripping', 'dripped',
    'overflow', 'overflowed',
    'backflow', 'drenched',
    'soggy', 'flooded', 'seeping'
])

# ── RISK / CONTAMINATION KEYWORDS ─────────────────────────────────────────
# Detects posts mentioning safety hazards or product contamination

RISK_KEYWORDS = [
    'foreign object', 'contamination', 'rust', 'rust water',
    'adhesive', 'mold', 'mould', 'insect', 'bug', 'fly',
    'metal particle', 'metal fragment',
    'hazardous substance', 'carcinogen', 'toxic',
    'VOC', 'benzene', 'toluene', 'styrene', 'xylene', 'TVOC'
]


In [ ]:
import ast
from googletrans import Translator

# Date range for this batch (update before each run)
start = (datetime.now() - timedelta(days=15)).strftime('%Y-%m-%d')
end   = (datetime.now() - timedelta(days=9)).strftime('%Y-%m-%d')
print(f'Processing: {start} to {end}')

In [ ]:
# ── DATABASE CONNECTION ──────────────────────────────────────────────────
# Set environment variables: SNOWFLAKE_USER, SNOWFLAKE_PASSWORD, SNOWFLAKE_ACCOUNT,
#                             SNOWFLAKE_DATABASE, SNOWFLAKE_SCHEMA, SNOWFLAKE_WAREHOUSE, OPENAI_API_KEY

con = snowflake.connector.connect(
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE']
)
cur = con.cursor()
translator = Translator()

## Part 1: Keyword Extraction
GPT-4 extracts top 3 satisfaction and dissatisfaction keywords per batch of reviews.

In [ ]:
# ── LOAD REVIEW DATA ─────────────────────────────────────────────────────

review = pd.DataFrame(cur.execute(f"""
    SELECT DATE, max(BRAND), max(SUBBRAND), REVIEW
    FROM REVIEW_TABLE
    WHERE DATE >= '{start}' AND DATE <= '{end}'
    GROUP BY DATE, REVIEW, PROPERTY;
"""))
review.columns = ['DATE', 'BRAND', 'SUBBRAND', 'REVIEW']

# Internal product sub-brands to analyze
TARGET_SUBBRANDS = [
    'MaxDry', 'CleanBebe', 'MagicPants', 'Bosongbosong', 'NatureBambu',
    'SwimmingPants', 'NatureMade', 'GoodNight', 'NatureMadeOrganic',
    'MagicSummer', 'GreenFinger', 'NatureSummer', 'PureCotton'
]

In [ ]:
# ── GPT-4 KEYWORD EXTRACTION ─────────────────────────────────────────────

openai.api_key = os.environ['OPENAI_API_KEY']
GPT_MODEL = 'gpt-4o'

def extract_keywords(reviews, brand):
    """Extract top 3 satisfaction and dissatisfaction keywords from a batch of reviews."""
    prompt = (
        f'Reviews: {reviews} '
        'Excluding delivery and sizing, identify 3 keywords each for '
        'satisfaction and dissatisfaction. '
        'Output format: {"satisfied": [kw1, kw2, kw3], "unsatisfied": [kw1, kw2, kw3]}. No other output.'
    )
    messages = [
        {'role': 'system', 'content': f'You are a marketer for the {brand} diaper brand.'},
        {'role': 'user', 'content': prompt}
    ]
    return openai.ChatCompletion.create(model=GPT_MODEL, messages=messages).choices[0].message.content

In [ ]:
# ── BATCH EXTRACTION ─────────────────────────────────────────────────────

col_sub, col_word, col_sentiment, col_date = [], [], [], []
dates = list(review.sort_values('DATE')['DATE'].unique())

for sub in TARGET_SUBBRANDS:
    for date_ in dates:
        rev = review[(review['SUBBRAND'] == sub) & (review['DATE'] == date_)]
        for i in range(int((len(rev) + 2) / 3)):
            try:
                str_d = extract_keywords(rev[['Rate', 'Text']][i*3:(i+1)*3].values, sub)
                d = ast.literal_eval(str_d)
                for sentiment in ['satisfied', 'unsatisfied']:
                    for keyword in d.get(sentiment, []):
                        col_sub.append(sub)
                        col_word.append(keyword)
                        col_sentiment.append(sentiment)
                        col_date.append(date_)
                print(f'Processed: {date_} | {sub}')
            except Exception as e:
                print(f'Failed: {e}')
                time.sleep(60)

output = pd.DataFrame({
    'DATE': pd.to_datetime(col_date, format='%Y%m%d').astype(str),
    'START_DT': start, 'END_DT': end,
    'SUBBRAND': col_sub,
    'KEYWORD': col_word,
    'SENTIMENT': col_sentiment
})

In [ ]:
# ── TRANSLATE KEYWORDS AND UPLOAD ────────────────────────────────────────
# Keywords are already extracted in English via GPT-4 prompt
# This step translates any remaining non-English terms

translation_map = {}
for kw in output['KEYWORD'].unique():
    try:
        if kw:
            translation_map[kw] = translator.translate(kw, src='auto', dest='en').text
        time.sleep(0.3)
    except Exception:
        translation_map[kw] = kw  # fallback to original if translation fails

output['KEYWORD_EN'] = output['KEYWORD'].map(translation_map)
output['SENTIMENT_EN'] = output['SENTIMENT']  # already in English

engine = create_engine(URL(
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE'],
))

with engine.connect() as con:
    output.to_sql(name='KEYWORDS_TABLE', con=con, if_exists='append',
                  method=pd_writer, index=False)
print(f'Uploaded {len(output)} keyword records')

## Part 2: Property Rating + Weekly Summary
Aggregates keywords into property-level scores (1-10) and generates a 2-sentence consumer summary.

In [ ]:
# ── PRODUCT PROPERTY DEFINITIONS ─────────────────────────────────────────

PROPERTIES = ['Absorbency', 'Softness', 'Comfort', 'Price', 'Hypoallergenic']

def make_score(pros, cons):
    """Score each property 1-10 based on extracted keywords."""
    prompt = (
        f'Pros: {pros}\nCons: {cons}\n'
        f'Score each of {PROPERTIES} out of 10 based on keyword frequency. '
        'Return only a list of 5 scores. No other output.'
    )
    messages = [
        {'role': 'system', 'content': 'You are a marketer for Huggies.'},
        {'role': 'user', 'content': prompt}
    ]
    return openai.ChatCompletion.create(model=GPT_MODEL, messages=messages).choices[0].message.content

def make_summary(pros, cons):
    """Generate a 2-sentence consumer opinion summary."""
    prompt = (
        f'Pros: {pros}\nCons: {cons}\n'
        'Summarize consumer opinion in 2 sentences based on keyword frequency.'
    )
    messages = [
        {'role': 'system', 'content': 'You are a marketer for Huggies.'},
        {'role': 'user', 'content': prompt}
    ]
    return openai.ChatCompletion.create(model=GPT_MODEL, messages=messages).choices[0].message.content

In [ ]:
# ── GENERATE SCORES AND SUMMARIES PER SUB-BRAND ──────────────────────────

df = pd.DataFrame(cur.execute('SELECT * FROM KEYWORDS_TABLE'))
df = df.rename(columns={1: 'date', 2: 'subbrand', 3: 'keyword', 4: 'sentiment'})
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d').astype(str)

rating_output = pd.DataFrame()

for sub in TARGET_SUBBRANDS:
    target = df[(df['date'] >= start) & (df['date'] <= end) & (df['subbrand'] == sub)]
    pros = target[target['sentiment'] == 'satisfied']['keyword'].tolist()
    cons = target[target['sentiment'] == 'unsatisfied']['keyword'].tolist()

    summary_text = make_summary(pros, cons).replace("'", '').replace('"', '')
    scores_raw = make_score(pros, cons)
    scores = list(map(float, scores_raw.replace('[','').replace(']','').split(',')))

    print(f'{start} - {end} | {sub}')

    for i, prop in enumerate(PROPERTIES):
        score_val = scores[i] if scores[i] != 0 else sum(scores) / len(scores)
        rating_output = rating_output.append({
            'START_DT': start, 'END_DT': end,
            'SUBBRAND': sub, 'PROPERTY': prop,
            'RATING': score_val, 'SUMMARY': summary_text
        }, ignore_index=True)

rating_output = rating_output.sort_values('START_DT')

with engine.connect() as con:
    rating_output.to_sql(name='RATING_SUMMARY_TABLE', con=con, if_exists='append',
                         method=pd_writer, index=False)
print(f'Uploaded {len(rating_output)} rating records')

## Part 3: Risk Keyword Detection
Scans all reviews for mentions of hazardous materials, contamination, or foreign objects.

In [ ]:
# ── RISK KEYWORD DETECTION ───────────────────────────────────────────────

risk_review = pd.DataFrame(cur.execute(f"""
    SELECT DATE, max(BRAND), max(SUBBRAND), REVIEW
    FROM REVIEW_TABLE
    WHERE DATE >= '{start}' AND DATE <= '{end}'
    GROUP BY DATE, REVIEW;
"""))
risk_review.columns = ['DATE', 'BRAND', 'SUBBRAND', 'REVIEW']

def detect_risk(review_text):
    """Return comma-separated risk keywords found in review."""
    return ', '.join(kw for kw in RISK_KEYWORDS if kw.lower() in review_text.lower())

risk_review['RISK_KEYWORD'] = risk_review['REVIEW'].apply(detect_risk)
risk_review['DATE'] = pd.to_datetime(risk_review['DATE'], format='%Y%m%d').astype(str)
risk_review['START_DT'] = start
risk_review['END_DT'] = end

with engine.connect() as con:
    risk_review.to_sql(name='RISK_TABLE', con=con, if_exists='append',
                       method=pd_writer, index=False)
print(f'Uploaded {len(risk_review)} risk records')